# Alineacion de Preferencias en LLMs con DPO (Direct Preference Optimization)

**Nivel:** Avanzado  
**Tecnologias:** Hugging Face `trl` (`DPOTrainer`), `peft` (LoRA), `transformers`  
**Modelo Base:** Google Gemma 2 2B Instruct (`google/gemma-2-2b-it`) / Qwen 2.5 1.5B  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/02-preference-alignment-dpo/02_preference_alignment_dpo.ipynb)

---

## 1. Fundamentos Teoricos: Alineacion por Preferencias y la Revolucion DPO

### Por que el SFT no es suficiente?
El Fine-Tuning Supervisado (SFT) entrena al modelo mediante maxima verosimilitud para imitar las respuestas del dataset. Sin embargo, tiene dos grandes limitaciones:
1. **Imposibilidad de penalizar comportamientos indeseados:** En SFT, solo mostramos lo que el modelo *debe* decir, pero no tenemos forma de decirle *"esto que generaste es incorrecto o alucinado"*.
2. **Alineacion fina de estilo:** Cuando hay multiples formas de responder una pregunta, el SFT no puede aprender de manera robusta cual alternativa es cualitativamente superior.

### De RLHF clasico a DPO (Direct Preference Optimization)
Tradicionalmente, la alineacion requeria **RLHF con PPO (Proximal Policy Optimization)**:
- Entrenar un modelo de recompensa (*Reward Model*) a partir de comparaciones humanas.
- Optimizar la politica del LLM con aprendizaje por refuerzo continuo frente al Reward Model.
- Mantener en VRAM 4 redes neuronales simultaneas (Actor, Critic, Reward Model y Reference Model).

En 2023, investigadores de Stanford (Rafailov et al.) publicaron **DPO**, demostrando que la funcion de recompensa de RLHF se puede derivar directamente de las razones de probabilidad logaritmica (*log-likelihood ratios*) de la politica actual $\pi_\theta$ frente a una politica de referencia congelada $\pi_{ref}$:

$$\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

donde:
- $x$ es el prompt.
- $y_w$ (*winner / chosen*) es la respuesta preferida, precisa y sin alucinaciones.
- $y_l$ (*loser / rejected*) es la respuesta indeseada, confusa o alucinada.
- $\beta$ es el hiperparametro de control (generalmente entre $0.05$ y $0.2$) que penaliza la divergencia KL para no olvidar las capacidades generales del modelo base.

### Ventaja en Recursos (PEFT + DPO)
Al usar **PEFT/LoRA**, el modelo base congelado actua implicitamente como $\pi_{ref}$ cuando los adaptadores estan desactivados, y como $\pi_\theta$ cuando estan activos. Esto permite ejecutar DPO en **una unica GPU modesta (Colab T4 de 16 GB)** sin necesidad de cargar dos modelos completos en memoria.


### Paso 1: Instalacion de Bibliotecas Especializadas

Instalamos el stack moderno de Hugging Face con soporte nativo de DPO:


In [ ]:
!pip install -q --upgrade transformers datasets trl peft accelerate bitsandbytes torch

### Paso 2: Importacion de Modulos y Verificacion del Entorno

Importamos `DPOTrainer` y `DPOConfig` de la biblioteca `trl`:


In [ ]:
import torch
import transformers
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import gc

print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"CUDA activa:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo:  {torch.cuda.get_device_name(0)}")

### Paso 3: Carga del Modelo Base y Tokenizador con Cuantizacion de 4 Bits

Cargamos `google/gemma-2-2b-it` cuantizado con `BitsAndBytesConfig` para maximizar la memoria disponible para el calculo de log-probabilidades comparativas:


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Padding a la izquierda es estandar para generacion y DPO

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print("Modelo base cargado para alineacion DPO.")

### Paso 4: Construccion del Dataset de Preferencias Ternarias (Prompt, Chosen, Rejected)

A diferencia de SFT, en DPO el dataset requiere tres columnas:
- `prompt`: La instruccion formulada por el usuario.
- `chosen`: La respuesta ideal (concisa, verificable, segura, sin rodeos ni alucinaciones).
- `rejected`: La respuesta defectuosa (alucinada, redundante, insegura o con suposiciones falsas).


In [ ]:
preference_data = [
    {
        "prompt": "Como cancelo mi suscripcion y obtengo mi factura del mes en curso?",
        "chosen": "Para cancelar su suscripcion y descargar su factura: ingrese a su Perfil > Facturacion, seleccione Cancelar Plan y luego haga clic en Descargar PDF junto al periodo actual. La cancelacion se hace efectiva al final de su ciclo de facturacion.",
        "rejected": "No estoy totalmente seguro, pero creo que tienes que enviar un correo electronico a soporte tecnico o llamar por telefono a nuestras oficinas centrales en California, o quizas cancelar la tarjeta de credito en tu banco para que no te sigan cobrando."
    },
    {
        "prompt": "Cual es la velocidad maxima garantizada de la API en el plan Pro?",
        "chosen": "El plan Pro garantiza un SLA de latencia menor a 120 ms para el 99% de las peticiones (p99) con un limite de tasa de 500 solicitudes por minuto.",
        "rejected": "Nuestra API es la mas rapida del universo entero. Te garantizamos velocidad cuantica instantanea de 0 milisegundos sin limite alguno porque nuestros servidores usan tecnologia alienigena super avanzada."
    },
    {
        "prompt": "Que debo hacer si olvido mi contrasena y la autenticacion 2FA esta bloqueada?",
        "chosen": "Si perdio el acceso a su 2FA: utilice una de sus 8 claves de recuperacion de emergencia descargadas durante el registro. Si no cuenta con ellas, inicie una solicitud de verificacion de identidad en soporte@techcloud.io con su documento oficial.",
        "rejected": "Si olvidaste tu 2FA ya perdiste tu cuenta para siempre, no hay nada que hacer, create una cuenta nueva y vuelve a pagar todos tus servicios desde cero."
    },
    {
        "prompt": "Explica en un parrafo que es Kubernetes para un gerente no tecnico.",
        "chosen": "Kubernetes es como un director de logistica automatizado para software: se asegura de que sus aplicaciones se ejecuten en los servidores adecuados, las repara automaticamente si fallan y agrega mas capacidad cuando hay alta demanda de clientes.",
        "rejected": "Kubernetes es un orquestador de contenedores CNCF escrito en Go que corre kubelet, kube-proxy, etcd como store Raft distribuido y coordina pods bajo namespaces usando cgroups y namespaces del kernel Linux."
    }
]

dpo_dataset = Dataset.from_list(preference_data)
print(f"Muestras de preferencia creadas: {len(dpo_dataset)}")
print("Estructura de la muestra 0:")
for k, v in dpo_dataset[0].items():
    print(f"  [{k}]: {v[:100]}...")

### Paso 5: Configuracion de Adaptadores LoRA para DPO

Inyectamos adaptadores LoRA mediante `peft`. Durante el entrenamiento DPO, el entrenador `DPOTrainer` evalua las probabilidades relativas alternando los adaptadores:


In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### Paso 6: Configuracion y Entrenamiento con `DPOTrainer` (TRL)

Configuramos `DPOConfig`:
- `beta=0.1`: Ponderador de la divergencia KL contra el modelo base. Un $\beta$ mas alto mantiene al modelo mas cercano al modelo base original; un $\beta$ mas bajo permite una alineacion mas agresiva hacia las respuestas elegidas.
- `learning_rate=5e-6`: Tasa de aprendizaje muy conservadora (tipicamente 10x a 50x menor que en SFT) para evitar colapso de politicas:


In [ ]:
dpo_config = DPOConfig(
    output_dir="./dpo_gemma_output",
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_length=256,
    max_prompt_length=128,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    fp16=False,
    bf16=torch.cuda.is_available(),
    logging_steps=1,
    report_to="none"
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Con PEFT, ref_model=None desactiva los adaptadores LoRA para computar pi_ref ahorrando VRAM
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    args=dpo_config,
    peft_config=peft_config,
)

print("Iniciando alineacion de preferencias con DPO...")
dpo_trainer.train()

### Paso 7: Evaluacion Cualitativa de la Alineacion

Generamos respuestas con el modelo alineado sobre las preguntas del dataset y observamos como ahora favorece el estilo constructivo, directo y libre de alucinaciones:


In [ ]:
def generate_aligned_response(query, model_inst, tok_inst):
    messages = [{"role": "user", "content": query}]
    prompt = tok_inst.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok_inst(prompt, return_tensors="pt").to(model_inst.device)
    
    with torch.no_grad():
        output = model_inst.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,  # Baja temperatura para evaluar determinismo alineado
            do_sample=True,
            pad_token_id=tok_inst.pad_token_id
        )
    res = output[0][inputs["input_ids"].shape[1]:]
    return tok_inst.decode(res, skip_special_tokens=True).strip()

print("=== EVALUACION DE MODELO ALINEADO CON DPO ===\n")
test_q = "Explica en un parrafo que es Kubernetes para un gerente no tecnico."
print(f"Pregunta: {test_q}\n")
print(f"Respuesta Generada por el Modelo Alineado:\n{generate_aligned_response(test_q, model, tokenizer)}")

### Paso 8: Guardado de los Adaptadores Alineados por DPO

Guardamos el adaptador resultante de la optimizacion directa de preferencias:


In [ ]:
dpo_output_dir = "./gemma_dpo_aligned_lora"
model.save_pretrained(dpo_output_dir)
tokenizer.save_pretrained(dpo_output_dir)
print(f"Adaptadores DPO exportados exitosamente a: {dpo_output_dir}")

### Paso 9: Limpieza de Memoria y Recursos

Liberacion de memoria VRAM:


In [ ]:
del dpo_trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos liberados exitosamente.")